# 4. Logistic Regression

표준화는 학습 데이터에만 맞추고, 클래스 가중치를 사용해 불균형 분류 성능을 평가합니다.

각 노트북은 학습/테스트 분할부터 독립 실행할 수 있습니다. 모델 전처리는 학습 데이터에만 맞춰 테스트 데이터 누수를 방지합니다.

## 공통 데이터 준비

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

data_dir = Path.cwd() / "data"
if not data_dir.is_dir():
    data_dir = Path.cwd().parent / "data"
if not data_dir.is_dir():
    raise FileNotFoundError("data 폴더를 찾을 수 없습니다. 프로젝트 루트 또는 notebooks 폴더에서 실행하세요.")

X_raw = pd.read_csv(data_dir / "secom.data", sep=r"\s+", header=None)
labels = pd.read_csv(data_dir / "secom_labels.data", sep=r"\s+", header=None)
y = labels.iloc[:, 0].astype(int)

if len(X_raw) != len(y):
    raise ValueError(f"센서 데이터와 라벨 행 수가 다릅니다: {len(X_raw)} != {len(y)}")

# 분할 후 학습 데이터에서만 결측률 기준과 중앙값을 학습해 데이터 누수를 막습니다.
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42, stratify=y
)
keep_columns = X_train_raw.columns[X_train_raw.isna().mean() < 0.8]
X_train_raw = X_train_raw.loc[:, keep_columns]
X_test_raw = X_test_raw.loc[:, keep_columns]

imputer = SimpleImputer(strategy="median")
X_train = pd.DataFrame(
    imputer.fit_transform(X_train_raw),
    columns=keep_columns,
    index=X_train_raw.index,
)
X_test = pd.DataFrame(
    imputer.transform(X_test_raw),
    columns=keep_columns,
    index=X_test_raw.index,
)

keep_columns = X_train.columns[X_train.nunique(dropna=False) > 1]
X_train = X_train.loc[:, keep_columns]
X_test = X_test.loc[:, keep_columns]

print(f"학습 데이터: {X_train.shape}, 테스트 데이터: {X_test.shape}")
print("학습 라벨 분포:")
print(y_train.value_counts().sort_index())


학습 데이터: (1253, 466), 테스트 데이터: (314, 466)
학습 라벨 분포:
0
-1    1170
 1      83
Name: count, dtype: int64


## 표준화 및 모델 학습

In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)

LABELS = [-1, 1]
TARGET_NAMES = ["PASS (-1)", "FAIL (1)"]


def report_results(y_true, y_pred):
    print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
    print(classification_report(
        y_true,
        y_pred,
        labels=LABELS,
        target_names=TARGET_NAMES,
        zero_division=0,
    ))
    print("Confusion matrix (actual rows, predicted columns):")
    print(confusion_matrix(y_true, y_pred, labels=LABELS))


scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42,
)
lr.fit(X_train_scaled, y_train)
pred_lr = lr.predict(X_test_scaled)

report_results(y_test, pred_lr)

Accuracy: 0.8376
              precision    recall  f1-score   support

   PASS (-1)       0.94      0.88      0.91       293
    FAIL (1)       0.11      0.19      0.14        21

    accuracy                           0.84       314
   macro avg       0.52      0.54      0.52       314
weighted avg       0.88      0.84      0.86       314

Confusion matrix (actual rows, predicted columns):
[[259  34]
 [ 17   4]]


In [3]:
from pathlib import Path
import pandas as pd

RESULTS_DIR = data_dir.parent / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


def normalize_labels(values):
    values = pd.Series(values).reset_index(drop=True)
    unique_values = set(values.dropna().unique())

    if unique_values.issubset({-1, 1}):
        values = values.map({-1: 0, 1: 1})

    return values.astype(int)


MODEL_NAME = "04Logistic_Regression"
y_true_saved = normalize_labels(y_test)
y_pred_saved = normalize_labels(pred_lr)

result_df = pd.DataFrame({
    "y_true": y_true_saved,
    "y_pred": y_pred_saved,
})

result_path = RESULTS_DIR / f"{MODEL_NAME}.csv"
result_df.to_csv(result_path, index=False)

print(f"저장 완료: {result_path.name}")
print(result_df.head())

저장 완료: 04Logistic_Regression.csv
   y_true  y_pred
0       0       0
1       0       1
2       0       0
3       0       0
4       0       0
